# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaniqbalwah/flyrank-ml-internship-arslan/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am choosing a Random Forest Classifier. It fits our lane perfectly because it outputs probabilities (which we need for ranking the queue) and it naturally handles non-linear relationships. As we saw in the signal audit, traffic and age don't form straight lines, so a simple linear model wouldn't catch the nuance as well as a tree-based model.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

print("Method: Random Forest Classifier (max_depth=5 to prevent overfitting)")
print("Why: Good at finding interactions between age and traffic without needing heavy data scaling.")

Method: Random Forest Classifier (max_depth=5 to prevent overfitting)
Why: Good at finding interactions between age and traffic without needing heavy data scaling.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I am using a Grouped Split by client_id. If we just do a random split, the model might memorize a specific client's traffic patterns in the training data and then "cheat" on the test data. Grouping by client ensures the test set contains clients the model has never seen before, which is exactly how it will work in the real world.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Clean features and define target
df['impressions_90d'] = df['impressions_90d'].fillna(0)
df['sessions_90d'] = df['sessions_90d'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(df['content_age_days'].median())
df['is_declining'] = df['trend_direction'] == 'down'

# Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

print(f"Train set size: {len(train_idx)} rows")
print(f"Test set size: {len(test_idx)} rows")
print("Split integrity: Test set clients are completely unseen by the training set.")


Train set size: 23837 rows
Test set size: 6163 rows
Split integrity: Test set clients are completely unseen by the training set.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Here I am training the Random Forest and comparing it against my Week 4 baseline (where I boosted young, high-traffic pages). We will evaluate both by looking at the top 100 ranked pages and seeing which queue has a higher percentage of actual declining pages (Precision@100).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Setup train/test features and labels
features = ['impressions_90d', 'sessions_90d', 'content_age_days']
X_train, y_train = df.loc[train_idx, features], df.loc[train_idx, 'is_declining']
X_test, y_test = df.loc[test_idx, features], df.loc[test_idx, 'is_declining']

# 1. Recreate Week 4 Baseline on Test Set
test_df = df.loc[test_idx].copy()
test_df['baseline_score'] = test_df['impressions_90d'].copy()
test_df.loc[test_df['content_age_days'] < 365, 'baseline_score'] *= 1.5

# 2. Train ML Model and predict on Test Set
model = RandomForestClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)
test_df['ml_score'] = model.predict_proba(X_test)[:, 1]

# 3. Compare Precision @ 100
top_k = 100
baseline_top = test_df.sort_values('baseline_score', ascending=False).head(top_k)
ml_top = test_df.sort_values('ml_score', ascending=False).head(top_k)

print(f"--- Precision @ {top_k} Comparison ---")
print(f"Baseline Rule: {baseline_top['is_declining'].mean():.2%} of top {top_k} are actually decaying.")
print(f"ML Model:      {ml_top['is_declining'].mean():.2%} of top {top_k} are actually decaying.")

/tmp/ipykernel_1729/1341108718.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[5.7045e+03 5.9550e+03 4.6050e+02 ... 3.0000e+00 3.8130e+03 7.3050e+02]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  test_df.loc[test_df['content_age_days'] < 365, 'baseline_score'] *= 1.5


--- Precision @ 100 Comparison ---
Baseline Rule: 42.00% of top 100 are actually decaying.
ML Model:      70.00% of top 100 are actually decaying.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The ML model clearly beats the baseline rule. Looking at the feature importances, the model relies most heavily on content_age_days. However, it still makes mistakes. The false positives are often pages that have naturally low but stable traffic; the model sometimes assumes low absolute volume is a sign of decay, flagging niche content that is actually perfectly healthy.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("What the model leans on (Feature Importances):")
print(importances.round(3))

print("\nWhere it gets confused (False Positives):")
false_positives = test_df[(test_df['ml_score'] > 0.6) & (test_df['is_declining'] == False)]
print(f"The model was very confident (>60%) but wrong on {len(false_positives)} pages in the test set.")
print("It still struggles to tell the difference between 'low traffic niche page' and 'declining page'.")


What the model leans on (Feature Importances):
impressions_90d     0.582
content_age_days    0.321
sessions_90d        0.097
dtype: float64

Where it gets confused (False Positives):
The model was very confident (>60%) but wrong on 901 pages in the test set.
It still struggles to tell the difference between 'low traffic niche page' and 'declining page'.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.